# YOLOX Polygon Training: Thermal Cheetah Dataset

This notebook demonstrates how to train YOLOX with **polygon bounding box** support using the Thermal Cheetah dataset.

### Interactive Features
- **Real-time Loss Plots**: Training and Validation loss curves update every epoch.
- **Live Visualization**: See polygon predictions on validation images after each epoch.

In [ ]:
import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import cv2
from tqdm.notebook import tqdm

# 1. Setup paths
project_root = str(Path(os.getcwd()).absolute())
if project_root not in sys.path:
    sys.path.append(project_root)
os.environ['PYTHONPATH'] = f"{project_root}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

from yolox.exp import get_exp
from yolox.core import Trainer
from yolox.utils import setup_logger, vis
from yolox.utils.boxes import postprocess, postprocess_polygon
from yolox.data.data_augment import ValTransform

print(f"Project root: {project_root}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
def vis_poly(img, bboxes, scores, cls_ids, conf=0.5, class_names=None):
    """Visualize polygon predictions."""
    for i in range(len(bboxes)):
        score = scores[i]
        if score < conf:
            continue
        
        poly = np.array(bboxes[i], dtype=np.int32).reshape((-1, 2))
        cls_id = int(cls_ids[i])
        color = (0, 255, 0) # Green for predictions
        
        cv2.polylines(img, [poly], isClosed=True, color=color, thickness=2)
        
        text = f"{class_names[cls_id] if class_names else cls_id}: {score:.2f}"
        x, y = poly[0]
        cv2.putText(img, text, (int(x), int(y) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

class InteractiveTrainer(Trainer):
    def __init__(self, exp, args):
        super().__init__(exp, args)
        self.train_losses = []
        self.val_losses = []
        self.maps = []
        self.epochs = []
        self.class_names = ["cheetah", "person", "other"]

    def after_epoch(self):
        # Standard YOLOX after_epoch saves ckpt and potentially runs eval
        super().after_epoch()
        
        # Record training loss
        total_loss = self.meter["total_loss"].avg if "total_loss" in self.meter.keys() else 0
        self.train_losses.append(total_loss)
        self.epochs.append(self.epoch + 1)
        
        # Calculate Validation Loss explicitly
        val_loss = self.get_val_loss()
        self.val_losses.append(val_loss)
        
        # Get mAP from the evaluator results (already run in super().after_epoch if eval_interval is reached)
        # Or run it manually here if we want it every epoch regardless of exp.eval_interval
        ap50_95, ap50, summary = self.exp.eval(self.model, self.evaluator, self.is_distributed)
        self.maps.append(ap50)
        
        # Update plots and show visualization
        self.update_notebook_ui()

    @torch.no_grad()
    def get_val_loss(self):
        self.model.train() # YOLOX Head only computes loss in train mode
        total_val_loss = 0
        num_batches = 0
        val_loader = self.evaluator.dataloader
        
        for imgs, targets, _, _ in val_loader:
            imgs = imgs.to(self.device).to(self.data_type)
            targets = targets.to(self.device).to(self.data_type)
            
            outputs = self.model(imgs, targets)
            total_val_loss += outputs["total_loss"].item()
            num_batches += 1
            if num_batches >= 5: # Limit for speed
                break
                
        self.model.eval()
        return total_val_loss / num_batches if num_batches > 0 else 0

    def update_notebook_ui(self):
        clear_output(wait=True)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Plot Loss
        ax1.plot(self.epochs, self.train_losses, label='Train Loss', marker='o')
        ax1.plot(self.epochs, self.val_losses, label='Val Loss', marker='x', linestyle='--')
        ax1.set_title('Loss Curves')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.grid(True)
        ax1.legend()
        
        # Plot mAP
        ax2.plot(self.epochs, self.maps, label='mAP@50', color='green', marker='s')
        ax2.set_title('Validation mAP@50')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('mAP')
        ax2.grid(True)
        ax2.legend()
        
        plt.show()
        
        # Visualize internal predictions
        self.visualize_predictions()

    @torch.no_grad()
    def visualize_predictions(self, num_images=2):
        self.model.eval()
        val_loader = self.evaluator.dataloader
        
        for imgs, _, info_imgs, ids in val_loader:
            imgs = imgs.to(self.device).to(self.data_type)
            outputs = self.model(imgs)
            
            if self.exp.use_polygon:
                outputs = postprocess_polygon(outputs, self.exp.num_classes, self.exp.test_conf, self.exp.nmsthre)
            else:
                outputs = postprocess(outputs, self.exp.num_classes, self.exp.test_conf, self.exp.nmsthre)
            
            fig, axes = plt.subplots(1, min(num_images, len(imgs)), figsize=(15, 7))
            if num_images == 1:
                axes = [axes]
                
            for i in range(min(num_images, len(imgs))):
                img = imgs[i].cpu().numpy().transpose(1, 2, 0)
                img = np.ascontiguousarray(img, dtype=np.uint8)
                
                if outputs[i] is not None:
                    output = outputs[i].cpu().numpy()
                    bboxes = output[:, 0:8] if self.exp.use_polygon else output[:, 0:4]
                    scores = output[:, 4] * output[:, 5]
                    cls_ids = output[:, 6]
                    
                    class_names = getattr(self.exp, "class_names", None)
                    if self.exp.use_polygon:
                        img = vis_poly(img, bboxes, scores, cls_ids, self.exp.test_conf, class_names)
                    else:
                        img = vis(img, bboxes, scores, cls_ids, self.exp.test_conf, class_names)
                
                axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                axes[i].set_title(f"Epoch {self.epoch+1} - Val Sample")
                axes[i].axis('off')
            
            plt.tight_layout()
            plt.show()
            break
        self.model.train()

## 3. Run Training Loop

In [ ]:
class Args:
    def __init__(self):
        self.batch_size = 2 if device == "cpu" else 4
        self.devices = None
        self.experiment_name = "yolox_notebook_run"
        self.fp16 = False
        self.cache = None
        self.occupy = False
        self.logger = "tensorboard"
        self.ckpt = None
        self.resume = False
        self.start_epoch = None
        self.num_machines = 1
        self.machine_rank = 0
        self.dist_backend = "nccl"
        self.dist_url = None

exp = get_exp("exps/example/yolox_thermal_cheetah_poly.py")
exp.max_epoch = 10 
exp.print_interval = 1
exp.eval_interval = 1
exp.class_names = ["cheetah", "person", "other"]

args = Args()
trainer = InteractiveTrainer(exp, args)
trainer.train()